In [0]:
%run ./Common_utils

In [0]:
%run ./api_config

In [0]:
%run ./issues_config

In [0]:
%run ./Pipeline_Logic

In [0]:
%run ./sprints_config

In [0]:
CATALOG         = "jira_data"
TARGET_SCHEMA   = "generalized"
TARGET_DATABASE = f"{CATALOG}.{TARGET_SCHEMA}"
LOG_TABLE       = "pipeline_run_log"

JIRA_USER       = "rajdeep.matieda@gmail.com"
JIRA_API_TOKEN  = ""
auth            = (JIRA_USER, JIRA_API_TOKEN)
headers         = {"Accept": "application/json"}

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_DATABASE}")
spark.sql(f"USE CATALOG {CATALOG}")     
spark.sql(f"USE SCHEMA {TARGET_SCHEMA}")

In [0]:
results = {}  # shared memory between pipelines

configs = [
    PROJECTS_CONFIG,  # always first — issues depends on it
    ISSUES_CONFIG,    # depends on results["projects"]
     BOARDS_CONFIG,    # ← add
    SPRINTS_CONFIG,
]

for config in configs:
    try:
        run_pipeline(
            spark, config, auth, headers,
            base, version, endpoint,
            TARGET_DATABASE, LOG_TABLE,
            results       # ← passed so child can read parent df
        )
        

    except Exception as e:
        log_pipeline_run(
            spark,
            f"{TARGET_DATABASE}.{LOG_TABLE}",
            pipeline_name     = config["name"],
            status            = f"FAILED: {str(e)}",
            records_processed = 0,
            mode              = "append" 
        )
